## Notebook 1: Stellar Temperature Estimation from Photometric Colours

**Audience:** First-year PhD students in Astrophysics  
**Theme:** Statistical inference and regression with astronomical data


###### Astrophysical motivation

A central task in stellar astrophysics is inferring physical stellar properties from observed light.

Ideally, one would use spectroscopy, which provides detailed diagnostics such as:

- effective temperature $T_{\rm eff}$
- surface gravity $\log g$
- metallicity $[{\rm Fe/H}]$
- radial velocity
- rotational broadening
- chemical abundances

However, spectroscopy is observationally expensive:
it requires substantial telescope time, complex reduction pipelines, and cannot easily scale to the billions of sources observed by modern sky surveys.

By contrast, imaging surveys provide broadband photometry for enormous stellar samples
(e.g. SDSS, Pan-STARRS, Gaia, Rubin/LSST).

This leads to a central inference problem:

> Can stellar effective temperature be recovered from photometric colours alone?

Classically, this was addressed through empirical colour–temperature calibrations.
Machine learning generalises this framework by enabling:

- multicolour inference
- nonlinear mappings
- uncertainty-aware fitting
- regularisation
- objective model validation

In this sense, machine learning extends — rather than replaces — traditional astrophysical calibration methods.

###### Learning goals

By the end of this notebook, you should understand:

- why stellar colours trace temperature
- limitations of single-colour estimators
- linear vs nonlinear regression
- heteroscedastic observational noise
- inverse-variance weighting
- overfitting and regularisation
- residual-based model evaluation
- hidden-variable degeneracies
- train/test validation principles

###### Scientific perspective

Most astronomical regression problems can be viewed as:

> physical intuition + empirical calibration + statistical inference + rigorous validation

This notebook focuses primarily on the first half of that pipeline:
constructing and validating regression models for stellar parameter estimation.

## Imports

We use standard scientific Python libraries commonly employed in astrophysical data analysis and machine learning workflows.

These libraries provide tools for:
- numerical computation
- tabular catalogue manipulation
- visualization
- statistical analysis
- regression modelling
- model validation


In [ ]:
import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


## Reproducibility and readability

Fixing the random seed ensures that each run of the notebook produces the same catalogue and the same train/test split.

This is essential for debugging, teaching, and reproducible scientific analysis, where results must be comparable across runs and implementations.

In [ ]:
SEED = 42
np.random.seed(SEED)

# Optional advanced sklearn configuration (only needed if metadata routing is used later)
sklearn.set_config(enable_metadata_routing=True)

# Plot styling chosen for readability and colorblind accessibility
plt.style.use("tableau-colorblind10")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 12

## Load the Dataset

In [ ]:
import pandas as pd
import os

def load_and_validate_stellar_data(file_path):
    """
    Loads stellar photometry and performs multi-layer validation.
    """
    # 1. Systemic Error Handling (File Presence)
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Critical Error: The file '{file_path}' was not found.")

    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        raise IOError(f"Failed to parse CSV: {e}")

    # 2. Structural Validation (Column presence)
    # These are essential for color-temperature inference
    required_columns = ["Teff", "u_g", "g_r", "r_i", "i_z", "feh", "sigma_phot"]
    missing_cols = [col for col in required_columns if col not in df.columns]
    
    if missing_cols:
        raise KeyError(f"Dataset is missing required astronomical features: {missing_cols}")

    # 3. Physical Validation (Filtering Non-Physical Artifacts)
    initial_count = len(df)

    # Drop any rows that have NaN in key features
    df = df.dropna(subset=required_columns)

    final_count = len(df)
    
    if final_count == 0:
        raise ValueError("Data validation failed: No valid stellar samples remain after filtering.")
    
    print(f"Successfully loaded {final_count} stars ({initial_count - final_count} rows discarded).")
    return df

    

In [ ]:
# Execution
try:
    df = load_and_validate_stellar_data("../data/sdss_processed_colors_v1.csv")
except Exception as err:
    print(f"Pipeline Halted: {err}")


## Target transformation

In astronomy, stellar effective temperature is commonly expressed on a logarithmic scale. This is motivated by both physical interpretation and statistical convenience.

###### Hertzsprung–Russell diagram scaling

Many stellar properties (luminosity, radius, and colour) follow power-law relationships with temperature. As a result:

- A change from 3000 K to 4000 K corresponds to a major physical transition (e.g. M-dwarf to K-star)  
- A change from 23000 K to 24000 K represents a comparatively small shift in stellar properties  

Logarithmic scaling compresses large values and expands small values, reflecting this non-uniform physical sensitivity.

###### Relative vs absolute error

Machine learning models typically minimize mean squared error (MSE), which treats all absolute errors equally.

- In linear space:
  - A 500 K error is treated the same at 4000 K and at 40000 K, despite having very different physical significance  

- In log space:
  - The model effectively learns *relative (percentage) errors*  
  - An error of 0.02 dex corresponds to roughly a 4.6% multiplicative error, independent of temperature scale  

Here, “dex” refers to a decimal logarithmic unit (base-10 logarithm).

###### Distributional stability

Stellar temperature distributions in surveys are typically skewed, with a long tail toward very hot stars.

Applying $\log_{10}(T_{\rm eff})$:

- reduces skewness  
- stabilizes variance  
- often produces a more Gaussian-like target distribution  

This can improve optimization behaviour for many regression models.

Overall, logarithmic scaling encodes the fact that astrophysical relevance is often multiplicative rather than additive.

In [ ]:
# Temperature often behaves more smoothly in logarithmic space.
# - relative errors matter more than absolute Kelvin errors
# - distributions are often better behaved
# - multiplicative trends become additive

df["log_Teff"] = np.log10(df["Teff"])


## Exploratory Data Analysis (EDA)

Before building regression models, it is essential to inspect the dataset directly.

In scientific machine learning, exploratory data analysis (EDA) serves several purposes:

- verifying data quality
- identifying missing or non-physical values
- understanding feature distributions
- detecting correlations and degeneracies
- building physical intuition about the problem

For astronomical catalogues, EDA is especially important because survey data often contain:
- heteroscedastic uncertainties
- selection effects
- correlated observables
- outliers produced by instrumental or reduction artefacts

A model trained without understanding these properties may achieve apparently good metrics while still learning non-physical patterns.

In this notebook, EDA helps us answer several key questions:

- What is the temperature range of the stellar sample?
- Are the colour distributions physically reasonable?
- Which colours correlate most strongly with temperature?
- How large is the observational scatter?
- Are there obvious nonlinearities that motivate more flexible models?

The following diagnostics provide a first statistical and astrophysical overview of the catalogue before formal modelling begins.

In [ ]:
# Basic catalogue overview
print("Catalogue shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst rows:")
display(df.head())

In [ ]:
# Statistical summary of numerical columns
df.describe()

In [ ]:
# Check for missing values
missing = df.isnull().sum()

print("Missing values per column:")
print(missing)

In [ ]:
# Distribution of stellar temperatures

fig, ax = plt.subplots(figsize=(6,4))

ax.hist(df["Teff"], bins=40)

ax.set_xlabel(r"$T_{\rm eff}$ [K]")
ax.set_ylabel("Number of stars")
ax.set_title("Distribution of Stellar Effective Temperatures")

plt.tight_layout()
plt.show()

In [ ]:
# Distribution in logarithmic temperature space

fig, ax = plt.subplots(figsize=(6,4))

ax.hist(df["log_Teff"], bins=40)

ax.set_xlabel(r"$\log_{10}(T_{\rm eff})$")
ax.set_ylabel("Number of stars")
ax.set_title("Distribution of Logarithmic Temperature")

plt.tight_layout()
plt.show()

In [ ]:
# Inspect colour distributions

colour_features = ["u_g", "g_r", "r_i", "i_z"]
colour_titles = [r"$u-g$", f"$g-r$", r"$r-i$", r"$i-z$"]

fig, axes = plt.subplots(2, 2, figsize=(8, 6))

for ax, feature, title in zip(axes.flatten(), colour_features, colour_titles):
    ax.hist(df[feature], bins=40)
    ax.set_title(title)
    ax.set_xlabel("Colour")
    ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# Pairwise correlations between colours and temperature

corr_cols = ["u_g", "g_r", "r_i", "i_z", "log_Teff", "feh", "sigma_phot"]

corr_matrix = df[corr_cols].corr()

custom_labels = [r"$u-g$", r"$g-r$", r"$r-i$", r"$i-z$", r"$\log T_{eff}$", "feh", "$\sigma_{phot}$"]

plt.figure(figsize=(8, 6))

ax = sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f"
)

# Apply custom labels
ax.set_xticklabels(custom_labels, rotation=45, ha="right")
ax.set_yticklabels(custom_labels, rotation=0)

ax.set_title("Feature Correlation Matrix")

plt.tight_layout()
plt.show()


In [ ]:
# Pairplot for visual inspection of correlations and nonlinear structure

custom_labels = [r"$u-g$", r"$g-r$", r"$r-i$", r"$i-z$", r"$\log T_{eff}$"]

g = sns.pairplot(
    df[
        ["u_g", "g_r", "r_i", "i_z", "log_Teff"]
    ],
    corner=True,
    height=1.5,  # controls subplot size
    plot_kws={"s": 10, "alpha": 0.5}
)


# Apply custom axis labels
for i, ax in enumerate(g.axes.flatten()):
    if ax is not None:
        ax.set_xlabel(ax.get_xlabel())
        ax.set_ylabel(ax.get_ylabel())

# Rename axis labels properly
for i, var in enumerate(custom_labels):
    g.axes[-1, i].set_xlabel(var)   # bottom row x-labels
    g.axes[i, 0].set_ylabel(var)    # left column y-labels

plt.show()

In [ ]:
# Photometric uncertainty distribution

fig, ax = plt.subplots(figsize=(6,4))

ax.hist(df["sigma_phot"], bins=40)

ax.set_xlabel(r"$\sigma_{\rm phot}$")
ax.set_ylabel("Number of stars")
ax.set_title("Photometric Uncertainty Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# Relationship between photometric uncertainty and temperature

plt.figure(figsize=(6, 4))

plt.scatter(
    df["sigma_phot"],
    df["log_Teff"],
    s=8,
    alpha=0.4
)

plt.xlabel(r"$\sigma_{\rm phot}$")
plt.ylabel(r"$\log_{10}(T_{\rm eff})$")
plt.title("Temperature vs Photometric Uncertainty")

plt.tight_layout()
plt.show()

In [ ]:
# Metallicity distribution

fig, ax = plt.subplots(figsize=(6,4))

ax.hist(df["feh"], bins=40)

ax.set_xlabel(r"$[{\rm Fe/H}]$")
ax.set_ylabel("Number of stars")
ax.set_title("Metallicity Distribution")

plt.tight_layout()
plt.show()

## Colour–temperature relation

Even a single colour contains information about stellar temperature, but the relationship is nonlinear rather than perfectly linear.

The colour–temperature plot shows the mapping between an observable quantity (e.g. $g-r$) and the physical parameter of interest ($T_{\rm eff}$).

###### The physics: main sequence trend

- **Hot stars (high $\log_{10} T_{\rm eff}$)**: have low (bluer) $g-r$ values  
- **Cool stars (low $\log_{10} T_{\rm eff}$)**: have high (redder) $g-r$ values  

This reproduces the familiar main-sequence colour–temperature trend.

###### Curvature of the relation

Due to nonlinear physical effects in stellar atmospheres and filter responses, the relation is not perfectly linear. This curvature is visually apparent in the scatter plot and indicates that simple linear regression will be insufficient for optimal accuracy.

###### Scatter in the relation (data “width”)

In an idealised universe, all points would lie on a single curve. In practice, the relation forms a band whose thickness is caused by:

- **Metallicity $[{\rm Fe/H}]$:** shifts colours through line-blanketing effects  
- **Reddening $E(B-V)$:** systematically moves stars toward redder colours  
- **Photometric noise $\sigma_{\rm phot}$:** introduces random scatter, especially for faint stars  

These combined effects produce the observed spread in the colour–temperature relation.

In [ ]:
# Metallicity effects on colour-temperature relation

plt.figure(figsize=(7, 5))

scatter = plt.scatter(
    df["g_r"],
    df["log_Teff"],
    c=df["feh"],
    s=8,
    alpha=0.6,
    cmap="viridis"
)

plt.xlabel("g - r")
plt.ylabel(r"$\log_{10}(T_{\rm eff})$")
plt.title("Metallicity Broadening of the Colour–Temperature Relation")

cbar = plt.colorbar(scatter)
cbar.set_label(r"$[{\rm Fe/H}]$")

plt.tight_layout()
plt.show()

## Colour–colour diagrams

In astronomy, colour–colour diagrams show that stellar populations are strongly organised by temperature, while dust and metallicity introduce scatter around this structure.

Unlike previous plots that relate an observable to a physical quantity (e.g. $g-r$ vs $T_{\rm eff}$), this diagram compares two observables directly.

###### The stellar locus

In an idealised, noise-free case, stars would lie along a narrow curved track known as the *stellar locus*.  
The position of a star along this curve is primarily determined by its effective temperature.

###### Colour encoding

By colouring points (e.g. using a viridis colormap), a third dimension is encoded into the 2D plot:

- **Cool stars (purple):** cluster toward the upper-right (large $g-r$, large $u-g$)  
- **Hot stars (yellow/green):** cluster toward the lower-left (small $g-r$, small $u-g$)  

This provides a visual confirmation that temperature is the dominant parameter shaping the locus.

###### Degeneracies and scatter

The locus is not infinitely thin. Its width reflects astrophysical and observational effects:

- Residual extinction uncertainties and calibration systematics can shift stellar colours 
- Metallicity affects different bands unequally, particularly shifting $u-g$ more strongly than $g-r$, broadening the sequence  

These effects demonstrate why a single colour is insufficient and why multi-dimensional colour information is required to break degeneracies.

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter( df["g_r"], df["u_g"], c=df["Teff"], s=8, alpha=0.6, cmap="viridis")
plt.ylabel("u - g")
plt.xlabel("g - r")
plt.title("Colour–Colour Diagram")
plt.colorbar(label=r"$T_{eff}$ [K]")
plt.tight_layout()
plt.show()


### Interpretation

Several important features are already visible in the dataset:

- The temperature distribution is non-uniform and mildly skewed
- Colour indices are strongly correlated with temperature
- The colour–temperature relation is nonlinear
- Metallicity broadens the stellar locus
- Photometric uncertainties vary significantly across the sample

These observations motivate several later modelling choices:

- logarithmic target transformation
- multicolour regression
- nonlinear feature engineering
- inverse-variance weighting
- regularisation

EDA therefore provides both statistical diagnostics and astrophysical intuition before formal machine learning begins.

## Train / test split

We split the dataset into:

- **Training set:** used to fit model parameters  
- **Test set:** held back until the end for unbiased evaluation  

###### Why this is necessary

The goal is to prevent overfitting. A model can simply memorise 9,000 stars without learning any underlying physical relationship.

Such memorisation would give excellent training performance but poor generalisation to new, unseen stars.

By evaluating on a held-out test set, we verify whether the model has learned a physically meaningful mapping between colours and temperature.

###### Target variable (y)

We use $\log_{10}(T_{\rm eff})$ as the target:

- This ensures that errors are interpreted in relative (percentage) terms  
- A 5% error at 4000 K is treated similarly to a 5% error at 8000 K  

###### Feature set (X)

We provide the model with four colour indices:

- $u-g$, $g-r$, $r-i$, $i-z$

While a single colour gives a rough estimate of temperature, multiple colours allow the model to disentangle effects such as dust reddening and metallicity, improving robustness and reducing degeneracies.

In [ ]:
# Four colours help break degeneracies caused by dust and metallicity.

features = ["u_g", "g_r", "r_i", "i_z"]
X = df[features]
y = df["log_Teff"]


In [ ]:
# test_size=0.25: You are hiding 25% of your stars in the test set
# random_state=SEED: This makes the experiment reproducible and we get the exact same stars in their training set.

X_train, X_test, y_train, y_test, sig_train, sig_test = train_test_split(
    X,
    y,
    df["sigma_phot"],
    test_size=0.25,
    random_state=SEED
)

print("Training stars:", len(X_train))
print("Test stars:", len(X_test))


## Utility functions

To compare regression models objectively, we define a small set of standard evaluation metrics.

In scientific machine learning, evaluating a model is as important as fitting it:
- a model with low training error may still generalise poorly to unseen data.

We define standard regression metrics to compare model performance.

###### RMSE (Root Mean Squared Error)

RMSE penalises large errors more strongly because residuals are squared before averaging.

- Highly sensitive to outliers  
- If a model performs well on most stars but fails on a few extreme cases (e.g. highly reddened or noisy stars), RMSE will increase significantly  
- For Gaussian errors, RMSE is directly related to the standard deviation of the residuals  

###### MAE (Mean Absolute Error)

MAE measures the typical magnitude of prediction errors.

- More robust to outliers than RMSE  
- Interpreted as the expected absolute error for a random star in the sample  
- A large gap between RMSE and MAE can indicate the presence of a small number of severe prediction failures  

###### $R^2$ (Coefficient of determination)

$R^2$ measures the fraction of variance in the target explained by the model:

$$R^2 = 1 - \frac{SS_{\text{res}}}{SS_{\text{tot}}} = 1 -  \frac{\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}{ \sum_{i=1}^{n} (y_i - \bar{y})^2} $$

That is 

*   **$R^2 = 1$:** The model explains all the variability of the response data around its mean (a perfect fit).
*   **$R^2 = 0$:** The model performs no better than simply predicting the mean of the dataset.
*   **$R^2 < 0$:** The model is performing worse than the horizontal mean line (usually an indication that the model is severely mis-specified or the data is non-linear).


In [ ]:
def evaluate_model(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "Model": name,
        "RMSE [dex]": rmse,
        "MAE [dex]": mae,
        "R2": r2
    }


## Naive baseline

In machine learning, it is essential to compare any model against a simple baseline before trusting more complex approaches.

As a baseline, we ignore all photometric features ($u$, $g$, $r$, $i$, $z$) and any physical modelling. Instead, we compute the mean temperature of the training set and predict this constant value for all test samples.

This is equivalent to assuming that every star has the same effective temperature, regardless of its observed properties.

In astronomy, such baselines are useful for assessing whether new data or models provide real predictive power. If a model using multiple photometric bands performs no better than this trivial predictor, it suggests one of the following:

- The photometric data are too noisy to contain useful signal  
- The target quantity is weakly or non-trivially correlated with the available features  
- The modelling approach is incorrect or underpowered  

In this sense, the mean baseline acts as a control experiment. It provides a reference point against which all subsequent models (e.g. linear regression or ensemble methods) must demonstrate improvement to be considered meaningful.


Notice that while $R^2 \approx 0$, the baseline still produces a relatively large RMSE.

This RMSE reflects the intrinsic spread of temperatures in the dataset rather than any predictive ability of the model.

In fact, for this constant predictor, the RMSE is directly related to the standard deviation of the target distribution (i.e. the variability of stellar temperatures in the catalogue).

In [ ]:
# y_train.mean(): Finds the average log 10 (T eff) of the training data.
y_mean = y_train.mean()

# np.repeat(..., len(y_test)): Creates a list for the test set where every entry is that average value.
baseline_pred = np.repeat(y_mean, len(y_test))

print(pd.Series(
    evaluate_model("Mean baseline", y_test, baseline_pred)
))

## Classical Single-Colour Calibration

In traditional empirical astronomy, stellar effective temperature ($T_{\rm eff}$) is often estimated using a single colour index via linear calibration. 

Historically, this approach was essential because many early surveys provided only two photometric bands (e.g., $B$ and $V$), where the $(B - V)$ index served as a fundamental "stellar thermometer."

Using modern SDSS filters, we model the relationship between the $g - r$ colour and temperature as a simple 1D linear regression:

$$\log_{10}(T_{\rm eff}) = m \cdot (g - r) + c$$

Our goal is to find the parameters $m$ (slope) and $c$ (intercept) that minimize the **Mean Squared Error (MSE)**. 

Given a dataset of $n$ observations $y_i = \log_{10}(T_{\rm eff})$ and predictions $\hat{y}_i$:

$$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

We define the **design matrix** $X$ (containing the feature $g-r$ and a column of ones for the intercept) and the **weight vector** $w = [m, c]^T$. 

The predictions are then $\hat{y} = Xw$, and the cost function $J(w)$ becomes:

$$J(w) = \| y - Xw \|^2 = y^T y - 2w^T X^T y + w^T X^T X w$$

Notice that the **gradient** is:

$$\nabla_w J(w) = -2X^T y + 2X^T X w$$

while the  **Hessian matrix** ($\mathbf{H}$) is **positive semi-definite (PSD)**.

$$\mathbf{H} = \frac{\partial^2 J}{\partial w^2} = 2X^T X \Longrightarrow z^T \mathbf{H} z \geq 0 \qquad \forall z\neq 0$$

The optimization problem is **convex** and

*   Any point where the gradient is zero is a **local minimum**.
*   Every local minimum is also the **global minimum**.

To find the optimal weights $w$, we set the gradient to zero and solve:

$$-2X^T(y - Xw) = 0 \implies X^T X w = X^T y$$

Performing a SVD decomposition $X = U \Sigma V^T$ where:

- $U$ is an orthogonal matrix containing left singular vectors. 
- $V^T$ is the transpose of the orthogonal matrix containing right singular vectors.
- $\Sigma$ is a rectangular diagonal matrix containing singular values $\sigma _{i}$ in descending order

we get the **Normal Equation**

$V \Sigma^T U^T U \Sigma V^T w =  V \Sigma^T U^T y \qquad \Longrightarrow \qquad w = V \Sigma^+ U^T y$

where $\Sigma^+$ is created by taking the reciprocal of every **non-zero** singular value and leaving the zeros as zeros.


> For ordinary least squares, the loss surface is convex, so any local minimum is also the global minimum.<br>This property does not generally hold for neural networks or other highly nonlinear models.


## From MSE to Likelihood: Statistical Interpretation

###### The Gaussian Noise Assumption
In physical sciences, we assume that each observation $y_i$ is composed of a true underlying signal $\hat{y}_i$ and some stochastic noise $\epsilon_i$:

$$y_i = \hat{y}_i + \epsilon_i, \quad \epsilon_i \sim \mathcal{N}(0, \sigma^2)$$

This Gaussian assumption is physically motivated by the **Central Limit Theorem**: 
> in astronomy, the combination of independent noise sources—such as photon shot noise, atmospheric turbulence, and electronic thermal noise—tends toward a Normal distribution.

###### The Likelihood Function
Under this assumption, the probability density of observing a single data point $y_i$ is given by the Gaussian distribution.

For a dataset of $n$ independent and identically distributed (i.i.d.) observations, the **Total Likelihood** $\mathcal{L}(\theta)$ is the product of the individual probabilities:

$$\mathcal{L}(\theta) =  \prod_{i=1}^{n} p(y_i \mid x_i, \theta) =\prod_{i=1}^{n} \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(y_i - \hat{y}_i)^2}{2\sigma^2}\right)$$

To find the best model, we want to maximize the Likelihood. 

In practice, we minimize the **Negative Log-Likelihood (NLL)**. Taking the logarithm transforms the product into a sum and simplifies the exponent:

$$\text{NLL}(w) = -\ln \mathcal{L}(w) = -\sum_{i=1}^{n} \left[ -\frac{1}{2}\ln(2\pi\sigma^2) - \frac{(y_i - X_i w)^2}{2\sigma^2} \right] = \frac{n}{2}\ln(2\pi\sigma^2) + \frac{1}{2\sigma^2} \sum_{i=1}^{n} (y_i - X_i w)^2$$

Notice that the first term and the denominator $2\sigma^2$ are constants with respect to the weights $w$. 

Therefore, minimizing the NLL is equivalent to minimizing the sum of squared residuals:

$$\nabla_w \text{NLL}(w) \propto \nabla_w \sum_{i=1}^{n} (y_i - X_i w)^2$$

> **Minimizing the Mean Squared Error is exactly equivalent to finding the Maximum Likelihood Estimate (MLE) for a model with Gaussian noise.**


## Weighted Least Squares (WLS)

Standard linear regression treats every star as equally reliable. 

However, astronomical datasets often contain measurements with varying levels of precision due to differences in stellar brightness, exposure times, or atmospheric conditions. To account for this, we use **Weighted Least Squares**.

###### The Weighting Mechanism
We assign a weight $w_i$ to each observation based on its reported uncertainty $\sigma_i$:

$$w_i = \frac{1}{\sigma_i^2}$$

This choice is not arbitrary; it is the **inverse-variance weighting**. 

In a statistical sense, variance represents the "uncertainty" or "noise floor" of a measurement. 

By using the inverse, we ensure that:

*   **High-Precision Stars (Small $\sigma$):** Receive large weights, dominating the calculation of the slope ($m$) and intercept ($c$).
*   **Low-Precision Stars (Large $\sigma$):** Receive small weights, effectively "de-weighting" their contribution so that their noise does not bias the final calibration.

In WLS, we modify the Mean Squared Error to a **Weighted Mean Squared Error (WMSE)**. 

Instead of summing simple residuals, we scale each residual by its corresponding weight:

$$J(w) = \sum_{i=1}^{n} w_i (y_i - \hat{y}_i)^2$$

There are two primary reasons why this is the standard approach in astronomy:

1.  **Heteroscedasticity:** In real surveys, noise is rarely constant across the sample (a condition called *heteroscedasticity*). WLS "normalizes" the data, ensuring that a noisy measurement from a faint star doesn't pull the regression line away from the high-quality data points.
2.  **Maximum Likelihood with Varying Errors:** If we assume each star has its own Gaussian error $\epsilon_i \sim \mathcal{N}(0, \sigma_i^2)$, then minimizing the WMSE above is the mathematically exact **Maximum Likelihood Estimate (MLE)** solution only when the target uncertainties are Gaussian and known. Here we use photometric uncertainties as a practical proxy for observational reliability.

In matrix notation, if we define a diagonal weight matrix $W$ where $W_{ii} = 1/\sigma_i^2$, the optimal solution is still a closed-form "Normal Equation":

$$w = (X^T W X)^{-1} X^T W y$$

This is the version of the pseudoinverse that `scikit-learn` or other solvers use under the hood when you pass `sample_weight` to the model.

###### Errors-in-Variables
 
In formal regression theory, weights should represent the uncertainty in the **target** ($y$), not the **features** ($X$). 

> Using weights on $X$-uncertainties is a common "astronomical shortcut." While it correctly down-weights noisy observations, it technically ignores the fact that noise in the predictors (colours) biases the slope—a phenomenon known as **regression dilution**. For a fully rigorous treatment, one would use an **Errors-in-Variables** model or a Bayesian approach that treats the true colours as latent variables.


In [ ]:
# Avoid division by zero
sig_train_safe = np.clip(sig_train, 1e-4, None)
w_train = 1.0 / sig_train_safe**2
# Normalise weights (important for some models)
w_train = w_train / np.mean(w_train)


In [ ]:
lin1_model = LinearRegression()
lin1_model.fit(X_train[["g_r"]], y_train, sample_weight=w_train)


lin1_pred = lin1_model.predict(X_test[["g_r"]]) # This is a test of Generalization. 


print(pd.Series(
    evaluate_model("Linear (g-r only)", y_test, lin1_pred)
))

In [ ]:
# 1. Select the first test sample as a DataFrame (keeps feature names)
sample_df = X_test[["g_r"]].iloc[[0]] 
print("Training Sample:")
print(sample_df)
# 2. Predict directly
# No warning! No reshaping needed!

sample_model_pred = lin1_model.predict(sample_df)[0]
print(f"\nIntercept:\t{lin1_model.intercept_:.4f}")
print(f"Coefficients:\t{lin1_model.coef_}" )

# 3. Manual calculation (for the student's understanding)
# We still use .values here because math operations are cleaner with arrays
sample_manual_pred = (sample_df.values @ lin1_model.coef_) + lin1_model.intercept_
print(f"\nModel .predict():                     {sample_model_pred:.4f}")
print(f"Manual Calculation:                     {sample_manual_pred[0]:.4f}")
print(f"Actual Target (y_test):                 {y_test.iloc[0]:.4f}")

In [ ]:
plt.figure()

plt.scatter(X_test["g_r"],y_test,s=1,edgecolor="black",c="darkorange",label="data")

plt.scatter(X_test["g_r"],lin1_pred,s=1,color="yellowgreen",label="1D Linear Regression")

plt.xlabel(r"$g-r$")
plt.ylabel(r"$\log T_{eff}$")
plt.title("Linear Regression")
plt.legend()
plt.show()

When we predict on the test set, the model takes 750 previously unseen stars and estimates their $\log_{10}(T_{\rm eff})$ using only their $g-r$ values.

If the model performs well, this indicates that the $g-r$–temperature relation behaves as an approximately universal physical mapping within this observational dataset.

Although this single-colour model significantly outperforms the mean baseline, it still exhibits non-negligible errors for two main reasons:

- **Underfitting (model misspecification)**<br>The true colour–temperature relation is nonlinear.

  - A linear model cannot capture this curvature, leading to systematic errors at the hot and cool extremes of the temperature range.

- **Degeneracy (hidden variables)**<br>The model only uses $g-r$ and therefore cannot disentangle different physical effects that produce similar colours. <br>In particular, it cannot distinguish whether a star appears red because it is:


  - intrinsically cool  
  - affected by interstellar dust  
  - influenced by metallicity effects  

These unobserved variables introduce irreducible ambiguity in the mapping from colour to temperature.

## Multicolour linear regression

We extend the model from a single colour (“1D thermometer”) to a four-dimensional feature space using all available colour indices.

The model now uses $u-g$, $g-r$, $r-i$, and $i-z$ simultaneously as predictors of stellar temperature.

We assume a linear form:

$
\log_{10}(T_{\rm eff}) =
\beta_0 + \beta_1 (u-g) + \beta_2 (g-r) + \beta_3 (r-i) + \beta_4 (i-z)
$

Compared to the single-colour model, this approach provides additional constraints on the inverse problem.

A single colour (e.g. $g-r$) cannot distinguish between:
- intrinsically cool stars  
- stars affected by imperfect extinction corrections or metallicity variations 

By incorporating multiple bands, the model can partially disentangle these effects:
- residual calibration systematics and metallicity effects produce distinct multiband signatures  
- intrinsic stellar colour variations follow a different multiband structure  

Even in the absence of astrophysical degeneracies, additional features improve performance.

Each colour measurement carries independent observational noise, so combining multiple bands helps reduce the impact of random errors and improves robustness.

- $u-g$: highly sensitive to temperature and metallicity, but relatively noisy  
- $g-r$, $r-i$: provide a stable baseline for the main temperature trend  
- $i-z$: less affected by dust, helping constrain reddening effects  

When comparing to the single-colour model, RMSE should decrease and $R^2$ should increase, indicating that additional photometric information improves predictive power.

However, the model remains linear in the input space: it fits a hyperplane in four dimensions, whereas the true relation is nonlinear. 

As a result, it still cannot fully capture curvature in the colour–temperature relation.

In [ ]:
lin4_model = LinearRegression()
lin4_model.fit(X_train, y_train, sample_weight=w_train)

lin4_pred = lin4_model.predict(X_test)

print(pd.Series(
    evaluate_model("Linear (4 colours)", y_test, lin4_pred)
))


In [ ]:
# 1. Select the first test sample as a DataFrame (keeps feature names)
sample_df = X_test.iloc[[0]] 
print("Training Sample:")
print(sample_df)
# 2. Predict directly
# No warning! No reshaping needed!

sample_model_pred = lin4_model.predict(sample_df)[0]
print(f"\nIntercept:\t{lin4_model.intercept_:.4f}")
print(f"Coefficients:\t{lin4_model.coef_}" )

# 3. Manual calculation (for the student's understanding)
# We still use .values here because math operations are cleaner with arrays
sample_manual_pred = (sample_df.values @ lin4_model.coef_) + lin4_model.intercept_
print(f"\nModel .predict():                   {sample_model_pred:.4f}")
print(f"Manual Calculation:                   {sample_manual_pred[0]:.4f}")
print(f"Actual Target (y_test):               {y_test.iloc[0]:.4f}")

## Nonlinear regression: polynomial features

Linear regression is limited to learning hyperplanes in feature space. Polynomial feature expansion allows it to represent nonlinear relationships by introducing curvature and feature interactions.

We now introduce feature engineering via a pipeline.

This is the first model in the notebook capable of capturing the curvature of the colour–temperature relation more effectively.

##### The pipeline

A **pipeline** chains multiple preprocessing and modelling steps into a single object.

It ensures that:
- the same transformations are applied consistently during training and testing  
- steps occur in the correct order  
- data leakage is avoided  
- the code remains modular and reproducible  

###### Polynomial features (degree = 2)

`PolynomialFeatures(degree=2)` augments the original feature set by generating nonlinear combinations.

For two generic features \(a\) and \(b\), it produces:
- Original terms: \(a\), \(b\)  
- Quadratic terms: \(a^2\), \(b^2\)  
- Interaction terms: \(ab\)  

Applied to multiple colour indices, this allows a linear model to fit curved surfaces in higher-dimensional space.

###### Standardisation

Polynomial expansion can produce features with widely different numerical scales.

For example:
- \(2.0^2 = 4.0\)  
- \(0.1^2 = 0.01\)  

`StandardScaler()` rescales each feature to have zero mean and unit variance, improving numerical stability and ensuring no feature dominates purely due to scale.

###### Expected behaviour

This model typically yields:
- higher \(R^2\)  
- lower RMSE  

compared to linear models.

It combines:
- **multi-band information** (reducing reddening and metallicity degeneracies)  
- **nonlinear feature interactions** (capturing curvature in the colour–temperature relation)  

Among classical regression approaches, this is the most expressive model used so far in the notebook.

In [ ]:
poly2_model = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    # We tell the scaler to ignore the weights
    ("scale", StandardScaler().set_fit_request(sample_weight=False)),
    # We tell the regression to request/use the weights
    ("reg", LinearRegression().set_fit_request(sample_weight=True))
])

poly2_model.fit(X_train, y_train, sample_weight=w_train)

poly2_pred = poly2_model.predict(X_test)

print(pd.Series(
    evaluate_model("Polynomial degree 2", y_test, poly2_pred)
))


## Overfitting and regularization


So far, we have progressively increased model flexibility:
- from single-colour linear regression
- to multicolour regression
- to nonlinear polynomial feature expansions

Increasing model complexity generally improves the ability to fit astrophysical structure.

However, beyond a certain point, additional flexibility begins to fit random noise rather than the underlying physical relation.

###### Overfitting

More specifically, we now increase model complexity by allowing polynomial features up to degree 8.

However, higher degree polynomial model flexibility introduces a major risk: **overfitting**.

The model may begin to fit noise-specific fluctuations (“wiggles”) in the training set rather than learning the true underlying physical relationship.

That is models have enough flexibility to fit not only the underlying signal but also the observational noise.

This section introduces one of the central concepts in statistical learning:

> a good model must balance flexibility against generalisation.


###### Regularization

Regularisation methods, such as Ridge and Lasso regression, constrain model complexity while preserving predictive power.

- **Ridge regression:** applies smooth shrinkage to all coefficients  
- **Lasso regression:** can drive some coefficients exactly to zero, performing implicit feature selection  

This increases the model’s capacity to represent fine structure in the data, including subtle astrophysical effects such as spectral features.


In [ ]:
# 1. Setup the range of degrees
degrees = np.arange(1, 7)
train_rmse = []
test_rmse = []

# 2. Iterate and collect scores
for d in degrees:
    # Create the same pipeline used in your notebook
    model = Pipeline([
        ("poly", PolynomialFeatures(degree=d, include_bias=False)),
        ("scale", StandardScaler().set_fit_request(sample_weight=False)),
        ("reg", LinearRegression().set_fit_request(sample_weight=True))
    ])
    
    # Fit on training data
    model.fit(X_train, y_train, sample_weight=w_train)
    
    # Predict on both sets
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Calculate RMSE (Root Mean Squared Error)
    train_rmse.append(np.sqrt(mean_squared_error(y_train, y_train_pred)))
    test_rmse.append(np.sqrt(mean_squared_error(y_test, y_test_pred)))

# 3. Visualization
plt.figure(figsize=(6, 4))

plt.plot(degrees, train_rmse, marker='o', label='Training Error (Bias)',       color='tab:blue', linestyle='--')
plt.plot(degrees, test_rmse,  marker='s', label='Test Error (Generalization)', color='tab:red',  linewidth=2)

plt.yscale('log') # Log scale helps see the explosion of error at high degrees
plt.xlabel('Polynomial Degree (Model Complexity)')
plt.ylabel('RMSE [dex] (Log Scale)')
plt.title('The Bias-Variance Tradeoff in Stellar Temperature Estimation')
plt.xticks(degrees)
plt.legend()
plt.grid(True, which="both", ls="-", alpha=0.2)

plt.tight_layout()
plt.show()

## Bias–Variance Tradeoff

When we move beyond simple linear models to polynomial expansions, we encounter the **Bias–Variance Tradeoff**. We can decompose the expected prediction error of a model as follows:

$$\mathbb{E}[(y - \hat{f}(x))^2] = \text{Bias}[\hat{f}(x)]^2 + \text{Var}[\hat{f}(x)] + \sigma^2$$

*   **Low Degree (High Bias):** Simple models (e.g., linear) are too rigid. They make strong assumptions that cannot capture the underlying non-linear physics of stellar atmospheres.
*   **High Degree (High Variance):** Extremely flexible models fit the training data perfectly—including the random noise $\epsilon$. This leads to "wiggly" solutions that generalize poorly to new observations.

For a polynomial degree $d=4$ with 4 input colours, the expansion generates **69 features**, including high-order interactions like $(g-r)^2 (u-g)^2$ or $(g-r)^3 (u-g)$. Without constraints, a standard OLS model will assign a coefficient $w_j$ to every term, often resulting in massive, opposing weights that "chase" noise.

### Regularization

To mitigate overfitting in high-dimensional spaces, we modify our cost function $J(w)$ by adding a **penalty term** $R(w)$ that discourages overly complex models.

###### Ridge Regression ($L_2$ Regularization)
Ridge regression adds a penalty proportional to the **squared magnitude** of the coefficients:

$$J_{\text{Ridge}}(w) = \| y - Xw \|^2 + \alpha \| w \|_2^2 = \| y - Xw \|^2 + \alpha \sum_{j=1}^{p} w_j^2$$

*   **Shrinkage:** The parameter $\alpha$ controls the strength of the penalty. As $\alpha$ increases, coefficients are shrunk toward zero.
*   **Stability:** It prevents any single high-order term (e.g., $(u-g)^4$) from dominating the model unless it is consistently supported by the data across the entire training set.

###### Lasso Regression ($L_1$ Regularization)
Lasso (Least Absolute Shrinkage and Selection Operator) penalizes the **absolute value** of the coefficients:

$$J_{\text{Lasso}}(w) = \frac{1}{2n} \| y - Xw \|^2 + \alpha \| w \|_1 = \frac{1}{2n} \| y - Xw \|^2 + \alpha \sum_{j=1}^{p} |w_j|$$

*   **Sparsity:** Unlike Ridge, the $L_1$ penalty can force coefficients to be **exactly zero**.
*   **Feature Selection:** In our 70-feature expansion, Lasso automatically identifies the most physically relevant colour interactions and discards the redundant or noise-driven terms.

### The Necessity of Standardization

When using `StandardScaler`, we transform our features to have a mean of 0 and a variance of 1:
$$z = \frac{x - \mu}{\sigma}$$

Regularization is **scale-dependent**. Because the penalty is applied to the magnitude of $w_j$, a feature with a very small numerical range (like a narrow colour index) would naturally require a larger coefficient to impact the prediction. This larger coefficient would be unfairly penalized by Ridge or Lasso compared to a feature with a large range. 

**Standardization ensures a "level playing field,"** where the penalty $\alpha$ applies equally to the predictive power of every feature, regardless of its original units or scale.

In [ ]:
poly4_model = Pipeline([
    ("poly", PolynomialFeatures(degree=4, include_bias=False)),
    ("scale", StandardScaler().set_fit_request(sample_weight=False)),
    ("reg", LinearRegression().set_fit_request(sample_weight=True))
])


ridge4_model = Pipeline([
    ("poly", PolynomialFeatures(degree=4, include_bias=False)),
    ("scale", StandardScaler().set_fit_request(sample_weight=False)),
#    ("reg", Ridge(alpha=10).set_fit_request(sample_weight=True))
    ("reg", Ridge(alpha=5).set_fit_request(sample_weight=True))
])

lasso4_model = Pipeline([
    ("poly", PolynomialFeatures(degree=4, include_bias=False)),
    ("scale", StandardScaler().set_fit_request(sample_weight=False)),
    ("reg", Lasso(alpha=1e-4, max_iter=10000).set_fit_request(sample_weight=True))
])

poly4_model.fit(X_train, y_train, sample_weight=w_train)
ridge4_model.fit(X_train, y_train, sample_weight=w_train)
lasso4_model.fit(X_train, y_train, sample_weight=w_train)

poly4_pred  = poly4_model.predict(X_test)
ridge4_pred = ridge4_model.predict(X_test)
lasso4_pred = lasso4_model.predict(X_test)

poly4_score_train = poly4_model.score( X_train, y_train)
ridge4_score_train = ridge4_model.score(X_train, y_train)
lasso4_score_train = lasso4_model.score(X_train, y_train)

poly4_score_test = poly4_model.score( X_test, y_test)
ridge4_score_test = ridge4_model.score(X_test, y_test)
lasso4_score_test = lasso4_model.score(X_test, y_test)

for name, score in zip(["Linear", "Ridge", "Lasso"], [poly4_score_train, ridge4_score_train, lasso4_score_train]):
    print(f"Score on Train set for {name} Regression:\t{score}")

print()
for name, score in zip(["Linear", "Ridge", "Lasso"], [poly4_score_test, ridge4_score_test, lasso4_score_test]):
    print(f"Score on  Test set for {name} Regression:\t{score}")



In [ ]:
# Show which features Lasso kept
rigde_coeffs = ridge4_model.named_steps['reg'].coef_
rigde_total_features = len(rigde_coeffs)
rigde_non_zero = np.sum(~np.isclose(rigde_coeffs, 0))
print(f"Rigde kept {rigde_non_zero} out of {rigde_total_features} features.")

# Show which features Lasso kept
lasso_coeffs = lasso4_model.named_steps['reg'].coef_
lasso_total_features = len(lasso_coeffs)
lasso_non_zero = np.sum(~np.isclose(lasso_coeffs, 0))
print(f"Lasso kept {lasso_non_zero} out of {lasso_total_features} features.")


In [ ]:
results = pd.DataFrame([
    evaluate_model("Mean baseline", y_test, baseline_pred),
    evaluate_model("Linear (g-r only)", y_test, lin1_pred),
    evaluate_model("Linear (4 colours)", y_test, lin4_pred),
    evaluate_model("Poly deg2", y_test, poly2_pred),
    evaluate_model("Poly deg4", y_test, poly4_pred),
    evaluate_model("Ridge deg4", y_test, ridge4_pred),
    evaluate_model("Lasso deg4", y_test, lasso4_pred),
])

results = results.sort_values("RMSE [dex]")
results


In [ ]:
# 1D regression comparison using only g-r
colour_1D = "g_r"
X1_train = X_train[[colour_1D]]
X1_test  = X_test[[colour_1D]]

# Models - Using standard Pipeline syntax
lin_model = LinearRegression()

poly2_model = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scale", StandardScaler()),
    ("reg", LinearRegression())
])

poly12_model = Pipeline([
    ("poly", PolynomialFeatures(degree=12, include_bias=False)),
    ("scale", StandardScaler()),
    ("reg", LinearRegression())
])

ridge12_model = Pipeline([
    ("poly", PolynomialFeatures(degree=12, include_bias=False)),
    ("scale", StandardScaler()),
    ("reg", Ridge(alpha=10.0))
])

# Train
# To pass weights through a pipeline, use: stepname__sample_weight
lin_model.fit(X1_train, y_train)
poly2_model.fit(X1_train, y_train)
poly12_model.fit(X1_train, y_train)
ridge12_model.fit(X1_train, y_train)

# Smooth grid for plotting
g_r_grid = np.linspace(X[colour_1D].min(), X[colour_1D].max(), 500)
X_plot = pd.DataFrame({colour_1D: g_r_grid})

# Predictions
lin_pred_plot = lin_model.predict(X_plot)
poly2_pred_plot = poly2_model.predict(X_plot)
poly12_pred_plot = poly12_model.predict(X_plot)
ridge12_pred_plot = ridge12_model.predict(X_plot)

# RMSE Calculation Helper
def get_rmse(model, X, y):
    return np.sqrt(mean_squared_error(y, model.predict(X)))

# Plot
fig, ax = plt.subplots(figsize=(8, 6))

# Data points
ax.scatter(X1_train[colour_1D], y_train, s=10, alpha=0.15, color="tab:blue", label="Train")
ax.scatter(X1_test[colour_1D], y_test, s=10, alpha=0.15, color="tab:orange", label="Test")

# Plotting the fits
models_to_plot = [
    (lin_pred_plot, "tab:green", "-", "Linear", lin_model),
    (poly2_pred_plot, "tab:olive", "--", "Poly deg2", poly2_model),
    (poly12_pred_plot, "tab:pink", "-.", "Poly deg12 (Overfit)", poly12_model),
    (ridge12_pred_plot, "tab:red", ":", "Ridge deg12 (Regularized)", ridge12_model),
]

for pred, color, style, name, model in models_to_plot:
    train_rmse = get_rmse(model, X1_train, y_train)
    test_rmse = get_rmse(model, X1_test, y_test)
    ax.plot(g_r_grid, pred, color=color, linestyle=style, linewidth=2.5,
            label=f"{name} (Train={train_rmse:.4f}, Test={test_rmse:.4f})")

ax.set_xlabel(r"$g-r$ color")
ax.set_ylabel(r"$\log_{10}(T_{\rm eff})$")
ax.set_title("1D Regression Comparison: Complexity vs Generalization")
ax.set_ylim(3.5, 4.0)
ax.legend(frameon=True, fontsize='small', loc='upper right')

plt.tight_layout()
plt.show()

## Conclusions

In this notebook, we explored how stellar effective temperature can be inferred from broadband photometric colours using progressively more sophisticated regression models.

Starting from a naive mean predictor, we developed a sequence of increasingly expressive models:

1. single-colour linear calibration  
2. multicolour linear regression  
3. nonlinear polynomial regression  
4. regularised high-dimensional models  

This progression mirrors a common scientific workflow in astronomy:
begin with physically interpretable baseline models, then gradually introduce statistical complexity only when justified by the data.



## Main scientific lessons

###### Stellar colours contain strong temperature information

Even a simple colour index such as $g-r$ acts as an effective stellar thermometer because stellar spectra systematically shift with temperature.

However, the relation is:
- nonlinear
- broadened by metallicity and dust
- affected by observational noise

This naturally motivates multivariate and nonlinear modelling approaches.

###### Multiple colours help break degeneracies

Using several colour indices simultaneously improves predictive performance because different astrophysical effects leave different signatures across wavelength bands.

In particular, multicolour information helps distinguish:
- intrinsically cool stars
- dust-reddened hot stars
- metallicity-driven colour shifts

This is a fundamental example of how higher-dimensional feature spaces reduce ambiguity in inverse problems.

###### Flexible models improve accuracy — but introduce risk

Polynomial feature expansion allowed the regression model to capture curvature in the colour–temperature relation.

However, increasing model flexibility also increased the danger of overfitting:
the model may begin fitting noise-specific fluctuations rather than genuine astrophysical structure.

This illustrated the classical bias–variance tradeoff:
- overly simple models underfit
- overly flexible models generalise poorly

###### Regularisation improves generalisation

Ridge and Lasso regression demonstrated how statistical constraints can stabilise high-dimensional models.

Regularisation:
- suppresses unstable coefficients
- reduces sensitivity to noise
- improves robustness on unseen data

These ideas are central across modern machine learning and scientific inference.

###### Validation is essential

Throughout the notebook, train/test separation ensured that model performance was evaluated on previously unseen stars.

This is critical in scientific machine learning:
a model that memorises the training catalogue has little scientific value if it cannot generalise to new observations.

Reliable validation is therefore as important as model construction itself.
